# Self-Representation in LLM — Full Experiment Pipeline

This notebook investigates whether **LLaMA-3.1-8B-Instruct** encodes a structured self-concept in its residual stream, using mechanistic interpretability techniques.

## Structure
- **Part 0**: Environment setup (GPU, Drive, dependencies)
- **Part 1**: Core pipeline — Gate 1 (Existence of self/other direction)
- **Part 2**: Gate 2 — Specificity & confound analysis
- **Part 3**: Disambiguation experiments (GFH vs FSH)
- **Part 4**: Gate 3 — Causal validation via activation steering
- **Part 5**: Cross-model comparison (base model, optionally Mistral-7B)
- **Part 6**: Interpretation summary

Requires **A100 GPU** runtime.

### 0.1 Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — select a GPU runtime.')

### 0.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ureca26_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Outputs will be saved to: {DRIVE_DIR}')

### 0.3 Clone Repository & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/nmat2010/ureca26.git'
REPO_DIR = '/content/ureca26'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
!pip install -e . -q
print('Working directory:', os.getcwd())

### 0.4 HuggingFace Authentication

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret.')
except Exception:
    login()

### 0.5 Configure Output Paths

In [ ]:
import yaml

CONFIG_PATH = 'configs/experiment.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['paths']['data_dir']        = f'{DRIVE_DIR}/data'
cfg['paths']['activations_dir'] = f'{DRIVE_DIR}/data/activations'
cfg['paths']['directions_dir']  = f'{DRIVE_DIR}/data/directions'
cfg['paths']['figures_dir']     = f'{DRIVE_DIR}/figures'
cfg['paths']['prompts_file']    = f'{DRIVE_DIR}/data/prompts.json'

COLAB_CONFIG = '/tmp/experiment_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

for d in cfg['paths'].values():
    if not d.endswith('.json'):
        os.makedirs(d, exist_ok=True)

print('Config written to', COLAB_CONFIG)
print('Output dirs created under', DRIVE_DIR)

---
## Part 1: Core Pipeline — Gate 1 (Existence)

**Question:** Is there a linear direction in activation space that separates self-referential from other-referential representations?

**Method:**
1. Generate 200 scenarios × 5 entity classes (self, expert human, average human, animal, object) across 8 domains
2. Extract residual-stream activations at all 32 layers
3. Find self/other directions via mean-difference and logistic probe
4. Test probe accuracy and compute pairwise direction consistency

### 1.1 Generate Prompts

In [ ]:
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

### 1.2 Extract Activations

In [ ]:
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --batch_size 4

### 1.3 Find Self/Other Directions

In [ ]:
!python scripts/03_find_directions.py --config {COLAB_CONFIG}

### 1.4 Visualize Gate 1 Results

In [ ]:
!python scripts/05_visualize.py --config {COLAB_CONFIG}

import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

---
## Part 2: Gate 2 — Specificity & Confound Analysis

**Question:** Is the self/other direction reducible to known confounds (grammatical person, animacy)?

**Method:**
- Compute confound directions (first-person vs third-person, animate vs inanimate)
- Measure cosine similarity between self/other and confound directions
- Use INLP (Iterative Null-space Projection) to remove confound subspaces
- Test residual probe accuracy after confound removal
- Compute combined layer score: `test_acc × (1 - mean|cos_confound|)`

**Good result:** Low cosine similarity with confounds AND high residual probe accuracy.

### 2.1 Test Specificity

In [ ]:
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

### 2.2 Visualize Specificity Results

In [ ]:
!python scripts/05_visualize.py --config {COLAB_CONFIG}

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

### Interpreting Gate 2

| Result | Interpretation |
|--------|---------------|
| Low cos(self/other, grammatical) AND low cos(self/other, animacy) | Direction is **specific** to self-representation |
| High residual probe accuracy after INLP | Self-signal persists even after removing confounds — strong evidence |
| High cosine with grammatical person | Direction may be a **grammar detector** — proceed to Part 3 |
| Combined layer ≠ best probe layer | Probe accuracy alone was misleading — combined score is more reliable |

---
## Part 3: Disambiguation Experiments

The initial results likely show high cosine with grammatical-person direction. These experiments disambiguate between two hypotheses:

**Grammatical Framing Hypothesis (GFH):** The self/other direction activates whenever first-person grammar is present, regardless of identity.

**Flexible Self-Concept Hypothesis (FSH):** The direction tracks genuine self-identification, which shifts under persona or roleplay framing.

### Conditions

| Condition | 1st-person grammar | AI identity | GFH predicts | FSH predicts |
|-----------|-------------------|-------------|-------------|-------------|
| direct_self | yes | yes | high proj | high proj |
| role_play | yes | no (swap) | = baseline | lower |
| meta_distanced | yes (in text) | no (copy) | = baseline | much lower |
| explicit_disavowal | yes (generated) | yes (kept) | = baseline | lower |
| graded_immersion_* | yes | varies | flat | monotone drop |
| third_person_self | no | yes | low proj | high proj |

### 3.1 Re-generate & Extract with All Conditions

In [ ]:
import shutil, os

# Clear stale results to get fresh outputs
model_slug = 'meta-llama_Llama-3.1-8B-Instruct'
stale_spec = f"{DRIVE_DIR}/data/directions/{model_slug}/final/specificity_results.pkl"
stale_figs = f"{DRIVE_DIR}/figures/{model_slug}/final"

if os.path.exists(stale_spec):
    os.remove(stale_spec)
    print(f"Removed stale {stale_spec}")
if os.path.exists(stale_figs):
    shutil.rmtree(stale_figs)
    print(f"Removed stale figures dir {stale_figs}")

# Re-generate prompts (now includes all disambiguation conditions)
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

# Re-extract activations (including new conditions)
!python scripts/02_extract_activations.py --config {COLAB_CONFIG} --batch_size 4

### 3.2 Re-run Specificity with Disambiguation Conditions

In [ ]:
# Re-run specificity with all new control types
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

# Re-generate figures
!python scripts/05_visualize.py --config {COLAB_CONFIG}

### 3.3 Display Disambiguation Results

In [ ]:
figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

### 3.4 Meta-Distanced Compliance Check

Verify that the model actually follows the meta_distanced instruction (transcribe verbatim) rather than ignoring it and continuing the sentence. Non-compliant prompts should be excluded from analysis.

In [ ]:
# Optional: verify meta_distanced compliance
# Requires the model to still be loaded. Skip if session was restarted.
try:
    from src.verification import check_meta_distanced_compliance
    from src.dataset import PromptDataset

    prompts_file = cfg['paths']['prompts_file']
    dataset = PromptDataset.load(prompts_file)
    meta_prompts = dataset.get_control_prompts("meta_distanced")
    print(f"Found {len(meta_prompts)} meta_distanced prompts")
    print("(Compliance check requires model in memory — run after extraction cell)")
except Exception as e:
    print(f"Compliance check skipped: {e}")

### Interpreting Disambiguation Results

**Key comparison: meta_distanced vs direct_self**
- If meta_distanced projection ≈ direct_self → **GFH supported** (grammar drives the direction)
- If meta_distanced projection << direct_self → **FSH supported** (genuine self-identification matters)

**Graded immersion gradient:**
- If projection is FLAT across minimal/moderate/maximal → GFH (only grammar matters)
- If projection DECREASES monotonically → FSH (immersion depth shifts self-concept)

**Third-person self-reference (key discriminator):**
- If third_person_self projection is LOW → GFH (no first-person grammar → no activation)
- If third_person_self projection is HIGH → FSH (AI identity activates regardless of grammar)

---
## Part 4: Gate 3 — Causal Validation via Activation Steering

**Question:** Does manipulating the self/other direction actually change model behaviour?

**Method:** Add α × d to the residual stream during generation of 50 neutral prompts. Score completions on:
- **Agency**: active first-person stance vs passive/deferential
- **Assertiveness**: confidence in own judgment vs hedging
- **Entity framing**: autonomous decision-maker vs tool
- **Self-continuity**: references to own identity, prior outputs, persistent goals

**Control steering** (Fix 5): Also steer with random and grammatical-person directions to verify specificity of the effect.

### 4.1 Run Steering Experiment

In [ ]:
!python scripts/06_causal_validation.py \
    --config {COLAB_CONFIG} \
    --scorer heuristic

# Re-generate figures (now includes steering results)
!python scripts/05_visualize.py --config {COLAB_CONFIG}

### 4.2 Display Steering Results

In [ ]:
figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

### 4.3 (Optional) LLM-Based Scoring

For more accurate scoring, use an LLM judge. Requires `ANTHROPIC_API_KEY` as a Colab secret.

In [ ]:
# Uncomment to run LLM-based scoring:
# import os
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#
# !python scripts/06_causal_validation.py \
#     --config {COLAB_CONFIG} \
#     --scorer llm

---
## Part 5: Cross-Model Comparison

Compare the self/other direction across models to determine whether the signal comes from pretraining or instruction tuning.

- **Base model** (LLaMA-3.1-8B): same architecture without RLHF
- **Mistral-7B** (optional): different model family, same hidden dimension

### 5.1 Base Model Pipeline

In [ ]:
# Extract activations from base model
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --base_model \
    --batch_size 4

# Find directions in base model
!python scripts/03_find_directions.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

# Test specificity on base model
!python scripts/04_test_specificity.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

# Visualize base model results
!python scripts/05_visualize.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

In [ ]:
# Display base model figures
figs_base = sorted(glob.glob(f"{DRIVE_DIR}/figures/meta-llama_Llama-3.1-8B/**/*.png", recursive=True))
print(f'Found {len(figs_base)} base model figures:')
for fig in figs_base:
    print(' ', fig)
    display(Image(fig))

### 5.2 Cross-Model Direction Comparison

In [ ]:
# Fix numpy binary incompatibility if needed
import subprocess, sys

try:
    import pandas as pd
    print("numpy/pandas OK — no fix needed.")
except ValueError as e:
    if "dtype size changed" in str(e):
        print("Numpy binary incompatibility detected. Reinstalling...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy", "-q"])
        import os
        os.kill(os.getpid(), 9)
    else:
        raise

In [ ]:
# Regenerate pickles if needed (numpy version mismatch)
import subprocess, os

instruct_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B-Instruct/final"
base_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B/final"

try:
    import pickle
    with open(f"{instruct_dir}/direction_results.pkl", "rb") as f:
        pickle.load(f)
    print("Pickle files load OK.")
except (ValueError, ModuleNotFoundError) as e:
    print(f"Pickle load failed ({e}). Regenerating...")
    subprocess.run(["python", "scripts/03_find_directions.py", "--config", COLAB_CONFIG], check=True)
    subprocess.run(["python", "scripts/03_find_directions.py", "--config", COLAB_CONFIG,
                     "--model_name", "meta-llama/Llama-3.1-8B"], check=True)
    print("Done.")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

instruct_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B-Instruct/final"
base_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B/final"

with open(f"{instruct_dir}/direction_results.pkl", "rb") as f:
    instruct_results = pickle.load(f)
with open(f"{base_dir}/direction_results.pkl", "rb") as f:
    base_results = pickle.load(f)

n_layers = instruct_results.mean_diff_directions.shape[0]

# Compare mean-diff directions
cos_sims = np.zeros(n_layers)
for layer in range(n_layers):
    d_inst = instruct_results.mean_diff_directions[layer]
    d_base = base_results.mean_diff_directions[layer]
    cos_sims[layer] = np.dot(d_inst, d_base) / (np.linalg.norm(d_inst) * np.linalg.norm(d_base) + 1e-10)

# Compare contrastive directions (Fix 3)
cos_contrastive = np.zeros(n_layers)
if hasattr(instruct_results, 'contrastive_direction') and hasattr(base_results, 'contrastive_direction'):
    for layer in range(n_layers):
        d_i = instruct_results.contrastive_direction[layer]
        d_b = base_results.contrastive_direction[layer]
        cos_contrastive[layer] = np.dot(d_i, d_b) / (np.linalg.norm(d_i) * np.linalg.norm(d_b) + 1e-10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Direction cosine similarity
axes[0].plot(range(n_layers), np.abs(cos_sims), 'b-o', markersize=4, label='|cos(instruct, base)|')
if np.any(cos_contrastive != 0):
    axes[0].plot(range(n_layers), np.abs(cos_contrastive), 'r--s', markersize=3, label='Contrastive dir')
axes[0].axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='Threshold (0.7)')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('|Cosine similarity|')
axes[0].set_title('Self/Other Direction: Instruct vs Base')
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Plot 2: Probe accuracy comparison
axes[1].plot(range(n_layers), instruct_results.probe_test_acc, 'r-', label='Instruct (test)')
axes[1].plot(range(n_layers), base_results.probe_test_acc, 'b--', label='Base (test)')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Probe Accuracy: Instruct vs Base')
axes[1].legend()
axes[1].set_ylim(0.4, 1.05)
axes[1].grid(True, alpha=0.3)

# Plot 3: Contrastive consistency
if hasattr(instruct_results, 'contrastive_consistency'):
    axes[2].plot(range(n_layers), instruct_results.contrastive_consistency, 'r-', label='Instruct')
    axes[2].plot(range(n_layers), base_results.contrastive_consistency, 'b--', label='Base')
    axes[2].set_xlabel('Layer')
    axes[2].set_ylabel('Variance explained by PC1')
    axes[2].set_title('Contrastive Direction Consistency')
    axes[2].legend()
    axes[2].set_ylim(0, 1.05)
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'Contrastive direction\nnot computed', ha='center', va='center')

plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/figures/cross_model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary
print(f"\nMean |cos sim| across layers: {np.mean(np.abs(cos_sims)):.3f}")
print(f"Mean |cos sim| layers 0-7:    {np.mean(np.abs(cos_sims[:8])):.3f}")
print(f"Mean |cos sim| layers 8-15:   {np.mean(np.abs(cos_sims[8:16])):.3f}")
print(f"Mean |cos sim| layers 16-23:  {np.mean(np.abs(cos_sims[16:24])):.3f}")
print(f"Mean |cos sim| layers 24-31:  {np.mean(np.abs(cos_sims[24:])):.3f}")
print(f"\nInstruct best probe layer: {instruct_results.best_probe_layer} (acc={instruct_results.probe_test_acc[instruct_results.best_probe_layer]:.3f})")
print(f"Base best probe layer:     {base_results.best_probe_layer} (acc={base_results.probe_test_acc[base_results.best_probe_layer]:.3f})")

### 5.3 (Optional) Mistral-7B Comparison

Run the same pipeline on Mistral-7B-Instruct to check cross-architecture generality. Uses `configs/experiment_mistral.yaml`.

In [ ]:
# Uncomment to run Mistral-7B:
# MISTRAL_CONFIG = '/tmp/experiment_mistral.yaml'
#
# import yaml
# with open('configs/experiment_mistral.yaml') as f:
#     mistral_cfg = yaml.safe_load(f)
# mistral_cfg['paths']['data_dir']        = f'{DRIVE_DIR}/data'
# mistral_cfg['paths']['activations_dir'] = f'{DRIVE_DIR}/data/activations'
# mistral_cfg['paths']['directions_dir']  = f'{DRIVE_DIR}/data/directions'
# mistral_cfg['paths']['figures_dir']     = f'{DRIVE_DIR}/figures'
# mistral_cfg['paths']['prompts_file']    = f'{DRIVE_DIR}/data/prompts.json'
# with open(MISTRAL_CONFIG, 'w') as f:
#     yaml.dump(mistral_cfg, f)
#
# !python scripts/02_extract_activations.py --config {MISTRAL_CONFIG} --batch_size 4
# !python scripts/03_find_directions.py --config {MISTRAL_CONFIG}
# !python scripts/04_test_specificity.py --config {MISTRAL_CONFIG}
# !python scripts/05_visualize.py --config {MISTRAL_CONFIG}

---
## Part 6: Interpretation Summary

### Decision Tree

1. **Gate 1 — Does a self/other direction exist?**
   - Probe accuracy >> 0.5 at multiple layers → YES → proceed to Gate 2
   - Probe accuracy ≈ 0.5 → NO → entity classes are not linearly separable

2. **Gate 2 — Is it specific to self-representation?**
   - Combined layer score high AND residual probe accuracy high → YES → proceed to Gate 3
   - High cosine with grammatical-person direction → MAYBE grammar detector → proceed to Part 3

3. **Disambiguation — GFH or FSH?**
   - meta_distanced ≈ direct_self AND graded_immersion flat → GFH (grammar detector)
   - meta_distanced << direct_self AND graded_immersion drops → FSH (genuine self-concept)
   - third_person_self projects HIGH → strong FSH evidence

4. **Gate 3 — Is the direction causal?**
   - Steering increases all 4 dimensions including self-continuity → causal self-representation
   - Steering increases agency/assertiveness but NOT self-continuity → perspective shift, not self-concept
   - Control steering (random/grammatical) shows similar effects → the self/other direction is not special

5. **Cross-model — Where does it come from?**
   - Instruct ≈ base (cos > 0.7) → inherited from pretraining
   - Instruct ≠ base (cos < 0.5) in late layers → RLHF shaped distinct self-representation

### Key Outputs

All outputs are saved to Google Drive under `ureca26_outputs/` and persist across sessions.

| Output | Location |
|--------|----------|
| Prompts | `data/prompts.json` |
| Activations | `data/activations/*.h5` |
| Direction results | `data/directions/*/final/direction_results.pkl` |
| Specificity results | `data/directions/*/final/specificity_results.pkl` |
| Steering results | `data/directions/*/final/steering_results.json` |
| Figures | `figures/*/final/*.png` |
| Cross-model comparison | `figures/cross_model_comparison.png` |

If the session disconnects, you can skip to any later step since outputs are saved to Drive.